# OPERA Path A -- two-stage pretrained long-context study (train @ 512, eval @ 8192)

Pre-registered protocol: `docs/OPERA_PathA_prereg.md` (read it first).
This notebook is a thin wrapper: it clones the repo at a pinned commit,
sets credentials, and hands the session to `kaggle/patha_session.py`,
which auto-runs the next unfinished stage inside a wall-clock governor
(clean stop + checkpoint push before Kaggle's session cap).

**One-time setup (Kaggle web UI, outside this notebook):**
1. Settings -> Accelerator -> **GPU T4 x2** (never P100 -- confirmed broken).
2. Settings -> Internet -> **On**.
3. Add-ons -> Secrets: `KAGGLE_USERNAME` and `KAGGLE_KEY` (from your
   kaggle.com Settings -> API -> Create New Token).
4. After the first data-session push: Add Input ->
   `opera-lm-patha-data` and `opera-lm-patha-ckpt` (your datasets), so
   later sessions resume from the pushed checkpoints and mmap the pools.

**Each session:** Run All, walk away. Sessions continue server-side;
closing the tab is fine. The engine stops cleanly before the cap and
pushes everything. Rerun next session to continue the study.

In [ ]:
import os

REPO_URL = "https://github.com/Merna-Khalid/OPERA-LM"
COMMIT = "PATHA_COMMIT"          # pinned by the runbook; "main" also works

!rm -rf /kaggle/working/repo
!git clone $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
!git checkout $COMMIT
!pip install -q datasets tokenizers huggingface_hub kaggle

from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ["KAGGLE_USERNAME"] = sec.get_secret("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = sec.get_secret("KAGGLE_KEY")
os.environ.setdefault("GOVERNOR_HOURS", "10.5")
print("setup ok; governor hours:", os.environ["GOVERNOR_HOURS"])

## Session engine

`--stages auto` picks: env -> data (first, CPU-only session is fine) ->
smoke -> pretrain_opera -> pretrain_tf (pair) -> sft_opera -> sft_tf ->
curves. Force a stage with e.g. `--stages data` or inspect with
`--stages status`.

In [ ]:
!python kaggle/patha_session.py --stages auto 2>&1 | tail -100

In [ ]:
!python kaggle/patha_session.py --stages status